In [3]:
import pandas as pd
print("Python is working!")

Python is working!


In [4]:
from pathlib import Path
import pandas as pd

# Path to our original dataset
file_path = Path("../data/raw/online_retail_II.xlsx")

# If notebook runs from project root, adjust path
if not file_path.exists():
    file_path = Path("data/raw/online_retail_II.xlsx")

# Check file
print("Dataset found:", file_path.exists())

# Open Excel workbook without loading all records
excel_file = pd.ExcelFile(file_path)

print("\nAvailable worksheets:")
print(excel_file.sheet_names)

Dataset found: True

Available worksheets:
['Year 2009-2010', 'Year 2010-2011']


In [5]:
# Load both worksheets
df_2009 = pd.read_excel(
    excel_file,
    sheet_name="Year 2009-2010"
)

df_2010 = pd.read_excel(
    excel_file,
    sheet_name="Year 2010-2011"
)

print("2009-2010:", df_2009.shape)
print("2010-2011:", df_2010.shape)

# Combine both years
df = pd.concat(
    [df_2009, df_2010],
    ignore_index=True
)

print("Combined dataset:", df.shape)

display(df.head(10))

2009-2010: (525461, 8)
2010-2011: (541910, 8)
Combined dataset: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [6]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

Column names:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Data types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null 

Investigate Data Quality Issues


In [7]:
# 1. Dataset size
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

# 2. Missing values
print("\nMissing values:")
print(df.isna().sum())

# 3. Duplicate records
print("\nExact duplicates:", df.duplicated().sum())

# 4. Negative and zero quantities
print("\nNegative quantities:", (df["Quantity"] < 0).sum())
print("Zero quantities:", (df["Quantity"] == 0).sum())

# 5. Negative and zero prices
print("\nNegative prices:", (df["Price"] < 0).sum())
print("Zero prices:", (df["Price"] == 0).sum())

# 6. Cancelled invoices
cancelled = df["Invoice"].astype(str).str.startswith("C")

print("\nCancelled invoice rows:", cancelled.sum())

Total rows: 1067371
Total columns: 8

Missing values:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Exact duplicates: 34335

Negative quantities: 22950
Zero quantities: 0

Negative prices: 5
Zero prices: 6202

Cancelled invoice rows: 19494


Investigate Before Cleaning

A. Investigate Negative Quantities and Cancellations

In [8]:
# Identify cancelled invoices
df["IsCancelled"] = (
    df["Invoice"]
    .astype(str)
    .str.upper()
    .str.startswith("C")
)

# Compare negative quantities with cancellations
print("Negative quantity records:")
print((df["Quantity"] < 0).sum())

print("\nCancelled invoice records:")
print(df["IsCancelled"].sum())

print("\nNegative quantities with cancelled invoices:")
print(
    ((df["Quantity"] < 0) & df["IsCancelled"]).sum()
)

print("\nNegative quantities without cancelled invoices:")
print(
    ((df["Quantity"] < 0) & ~df["IsCancelled"]).sum()
)

# Display examples requiring investigation
display(
    df.loc[
        (df["Quantity"] < 0) & ~df["IsCancelled"]
    ].head(15)
)

Negative quantity records:
22950

Cancelled invoice records:
19494

Negative quantities with cancelled invoices:
19493

Negative quantities without cancelled invoices:
3457


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,False
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,False
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,False
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,False
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,False
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom,False
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom,False
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom,False
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom,False
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom,False


B. Investigate Missing Customer IDs

In [9]:
# Compare records with and without customer IDs
df["MissingCustomerID"] = df["Customer ID"].isna()

customer_quality = df.groupby(
    "MissingCustomerID"
).agg(
    Records=("Invoice", "size"),
    UniqueInvoices=("Invoice", "nunique"),
    TotalQuantity=("Quantity", "sum")
)

display(customer_quality)

# Examine examples
display(
    df[df["Customer ID"].isna()].head(10)
)

,Records,UniqueInvoices,TotalQuantity
MissingCustomerID,,,
False,824364,44876,10234128
True,243007,8752,374364


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled,MissingCustomerID
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom,False,True
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom,False,True
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom,False,True
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,False,True
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom,False,True
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom,False,True
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom,False,True
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom,False,True
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom,False,True
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom,False,True


C. Investigate Duplicate Transactions

In [10]:
# Examine duplicate groups without modifying the data
duplicate_records = df[
    df.duplicated(
        subset=[
            "Invoice",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "Price",
            "Customer ID",
            "Country"
        ],
        keep=False
    )
].copy()

print(
    "Rows belonging to duplicate groups:",
    len(duplicate_records)
)

display(
    duplicate_records.sort_values(
        ["Invoice", "StockCode"]
    ).head(20)
)

Rows belonging to duplicate groups: 67242


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,IsCancelled,MissingCustomerID
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,False,False
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,16329.0,United Kingdom,False,False
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,False,False
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,False,False
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,False,False
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,False,False
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,False,False
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,False,False
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,False,False
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,False,False


Classify Transaction Types

In [11]:
import numpy as np
import pandas as pd

# Use copies so our raw data stays unchanged
df["IsCancelled"] = (
    df["Invoice"]
    .astype(str)
    .str.upper()
    .str.startswith("C")
)

conditions = [
    # Cancelled invoice with negative quantity
    df["IsCancelled"] & (df["Quantity"] < 0),

    # Other cancelled invoice records
    df["IsCancelled"],

    # Negative quantity without cancellation prefix
    (df["Quantity"] < 0) & ~df["IsCancelled"],

    # Negative price
    df["Price"] < 0,

    # Zero price with non-negative quantity
    df["Price"] == 0,

    # Normal positive quantity and price
    (df["Quantity"] > 0) & (df["Price"] > 0)
]

categories = [
    "Cancelled Invoice - Negative Quantity",
    "Other Cancelled Invoice",
    "Negative Quantity - Investigation Required",
    "Negative Price - Investigation Required",
    "Zero Price Transaction",
    "Positive Sales Candidate"
]

df["TransactionType"] = np.select(
    conditions,
    categories,
    default="Other - Investigation Required"
)

transaction_summary = (
    df["TransactionType"]
    .value_counts()
    .rename_axis("Transaction Type")
    .reset_index(name="Record Count")
)

transaction_summary["Percentage"] = (
    transaction_summary["Record Count"]
    / len(df) * 100
).round(2)

display(transaction_summary)

,Transaction Type,Record Count,Percentage
0,Positive Sales Candidate,1041670,97.59
1,Cancelled Invoice - Negative Quantity,19493,1.83
2,Negative Quantity - Investigation Required,3457,0.32
3,Zero Price Transaction,2745,0.26
4,Negative Price - Investigation Required,5,0.00
5,Other Cancelled Invoice,1,0.00


In [12]:
sales_candidates = df[
    df["TransactionType"] == "Positive Sales Candidate"
].copy()

print("Positive sales records:", len(sales_candidates))

print(
    "Missing Customer IDs:",
    sales_candidates["Customer ID"].isna().sum()
)

print(
    "Missing descriptions:",
    sales_candidates["Description"].isna().sum()
)

print(
    "Duplicate occurrences:",
    sales_candidates.duplicated(
        subset=[
            "Invoice", "StockCode", "Description",
            "Quantity", "InvoiceDate", "Price",
            "Customer ID", "Country"
        ]
    ).sum()
)

print(
    "Unique invoices:",
    sales_candidates["Invoice"].nunique()
)

Positive sales records: 1041670
Missing Customer IDs: 236121
Missing descriptions: 0
Duplicate occurrences: 33757
Unique invoices: 40077


In [13]:
print("Dataset loaded:", "df" in globals())

if "df" in globals():
    print("Total records:", len(df))
    print("Columns:", df.columns.tolist())

Dataset loaded: True
Total records: 1067371
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'IsCancelled', 'MissingCustomerID', 'TransactionType']
